# 📊 EDA с визуализацией — Jupyter Notebook

Этот ноутбук — иллюстрированная версия урока 2.4 (EDA).
Здесь графики рендерятся **прямо на GitHub** при просмотре `.ipynb`.

Используем датасет клиентов из практики 1 модуля 02.
Если его ещё нет, запустите: `python3 практика_1_eda.py`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

# Путь к датасету относительно ноутбука
ROOT = Path('..').resolve()
CLIENTS = ROOT / 'datasets' / 'clients.csv'
TX = ROOT / 'datasets' / 'transactions.csv'

## 1. Загрузка и первый взгляд


In [ ]:
clients = pd.read_csv(CLIENTS, parse_dates=['registered_at'])
tx = pd.read_csv(TX, parse_dates=['ts'])

print(f'Клиентов: {len(clients):,}')
print(f'Транзакций: {len(tx):,}')
clients.head()

In [ ]:
clients.describe(include='all')

## 2. Распределение возраста

Классический график EDA — гистограмма ключевого числового признака.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(clients['age'], bins=30, kde=True, ax=ax[0])
ax[0].set_title('Распределение возраста клиентов')
ax[0].set_xlabel('Возраст')
ax[0].set_ylabel('Количество')

sns.boxplot(x='segment', y='age', data=clients, ax=ax[1])
ax[1].set_title('Возраст по сегментам')
plt.tight_layout()
plt.show()

## 3. Доход — сильный скос вправо


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(clients['monthly_income'], bins=50, ax=ax[0])
ax[0].set_title('Доход — оригинал (сильный скос)')
ax[0].set_xlabel('Доход, ₽')

sns.histplot(np.log1p(clients['monthly_income']), bins=50, ax=ax[1], color='orange')
ax[1].set_title('Доход — log1p (нормализован)')
ax[1].set_xlabel('log(1 + доход)')
plt.tight_layout()
plt.show()

print(f'Скос (skewness) до log: {clients["monthly_income"].skew():.2f}')
print(f'Скос (skewness) после log: {np.log1p(clients["monthly_income"]).skew():.2f}')

## 4. Сегменты и регионы


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

seg_counts = clients['segment'].value_counts()
ax[0].pie(seg_counts, labels=seg_counts.index, autopct='%1.1f%%', startangle=90)
ax[0].set_title('Распределение сегментов')

city_counts = clients[clients['city'] != '']['city'].value_counts().head(8)
city_counts.plot.barh(ax=ax[1])
ax[1].set_title('Топ-8 городов')
ax[1].set_xlabel('Кол-во клиентов')
plt.tight_layout()
plt.show()

## 5. Доход по сегментам и городам


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(x='segment', y='monthly_income', data=clients, ax=ax)
ax.set_title('Доход по сегментам — Violin plot')
ax.set_ylim(0, clients['monthly_income'].quantile(0.99))
plt.tight_layout()
plt.show()

## 6. Транзакции по времени


In [ ]:
tx['date'] = tx['ts'].dt.date
tx['hour'] = tx['ts'].dt.hour
tx['dow'] = tx['ts'].dt.dayofweek

fig, ax = plt.subplots(2, 2, figsize=(14, 9))

# Транзакции по дням
daily = tx.groupby('date').size()
daily.plot(ax=ax[0, 0], color='steelblue')
ax[0, 0].set_title('Транзакций по дням')
ax[0, 0].set_ylabel('Кол-во')

# По часам
tx.groupby('hour').size().plot.bar(ax=ax[0, 1], color='coral')
ax[0, 1].set_title('Транзакций по часам')
ax[0, 1].set_xlabel('Час')

# По дням недели
dow_names = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']
dow_data = tx.groupby('dow').size()
dow_data.index = [dow_names[i] for i in dow_data.index]
dow_data.plot.bar(ax=ax[1, 0], color='seagreen')
ax[1, 0].set_title('Транзакций по дню недели')

# Распределение сумм
sns.histplot(np.log1p(tx['amount']), bins=50, ax=ax[1, 1], color='purple')
ax[1, 1].set_title('log(1+сумма) транзакций')

plt.tight_layout()
plt.show()

## 7. Корреляции


In [ ]:
# Считаем агрегаты клиента: сколько транзакций, средняя сумма, etc.
client_aggr = tx.groupby('client_id').agg(
    tx_count=('amount', 'count'),
    avg_amount=('amount', 'mean'),
    max_amount=('amount', 'max'),
    sum_amount=('amount', 'sum'),
).reset_index()

df_corr = clients.merge(client_aggr, on='client_id')
num_cols = ['age', 'monthly_income', 'tx_count', 'avg_amount', 'max_amount', 'sum_amount']

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_corr[num_cols].corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Корреляции числовых признаков (Пирсон)')
plt.tight_layout()
plt.show()

## 8. Pairplot — всё со всем (на сэмпле)


In [ ]:
# На большом датасете pairplot будет медленным — берём сэмпл
sample = df_corr.sample(500, random_state=42)
sns.pairplot(
    sample[['age', 'monthly_income', 'tx_count', 'avg_amount', 'segment']],
    hue='segment',
    diag_kind='kde',
    plot_kws={'alpha': 0.4, 's': 20},
)
plt.suptitle('Pairplot: 500 клиентов', y=1.02)
plt.show()

## 9. Топ-категории транзакций


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

cat_counts = tx['category'].str.lower().str.strip().value_counts()
cat_counts.plot.bar(ax=ax[0])
ax[0].set_title('Кол-во транзакций по категориям')

cat_sums = tx.groupby(tx['category'].str.lower().str.strip())['amount'].sum().sort_values(ascending=False)
cat_sums.plot.bar(ax=ax[1], color='orange')
ax[1].set_title('Сумма транзакций по категориям')
ax[1].set_ylabel('Сумма, ₽')
plt.tight_layout()
plt.show()

## 10. Выбросы по сумме (boxplot + log scale)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
tx_with_seg = tx.merge(clients[['client_id', 'segment']], on='client_id')
sns.boxplot(x='segment', y='amount', data=tx_with_seg, ax=ax)
ax.set_yscale('log')
ax.set_title('Распределение сумм транзакций по сегментам (log scale)')
plt.tight_layout()
plt.show()

## 🧠 Выводы EDA

На этом учебном датасете:

1. Возраст распределён ~ нормально, медиана 40.
2. Доход — сильный правый скос (как у реальных финансовых данных).
3. ~70% клиентов в mass-сегменте, 5% — premium.
4. Премиум клиенты делают **больше** транзакций и с большими суммами.
5. Активность по часам: пик в дневные часы.
6. Дни недели: примерно равномерно (синтетика).
7. Сильная корреляция: monthly_income ↔ avg_amount (0.6+).

### Гипотезы для дальнейшего ML:
- Предсказывать сегмент по поведенческим признакам (без monthly_income).
- Детектить аномалии — суммы > 100× медианы клиента.
- Сегментировать клиентов через KMeans на поведенческих фичах.

Дальше → [Урок 2.5 — GroupBy, join, окна](./урок_5_groupby_join_windows.md)